In [1]:
import re, pymupdf, chromadb
from bisect import bisect_right
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

RUTA_NORMAS = '../data/Primera_y_Segunda_Division.pdf'
RUTA_REGLAS = '../data/Reglas de Juego 2026_27_paginas individuales.pdf'
INDICE_VIEJO = '../data/chroma_index'      # solo lectura: de aquí rescatamos los chunks de Wikipedia
INDICE_NUEVO = '../data/chroma_index_v2'
COLECCION_NUEVA = 'corpus_laliga_v2'

# 500 caracteres ~ 120 tokens: cabe en la ventana del modelo de embeddings (verifícalo en la celda siguiente).
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=60,
                                          separators=["\n\n", "\n", ". ", " "])

def normalizar_espacios(t: str) -> str:
    t = re.sub(r'[ \t]+', ' ', t)
    return re.sub(r'\n{3,}', '\n\n', t).strip()

def limpiar_reglas(t: str) -> str:
    t = re.sub(r'[\ue000-\uf8ff]', '', t)                    # glifo "tofu" de las viñetas (área privada de Unicode)
    t = re.sub(r'[\u2000-\u200f\u2028\u2029\x07]', ' ', t)    # espacios/controles invisibles
    # Quitamos el encabezado repetido de cada página. El capítulo NO se pierde: lo recupera el mapa de más abajo.
    return re.sub(r'Reglas de Juego \d{4}/\d{2}[^\n]*\n?', '', t)

/home/angel/sistema-multiagente-futbol/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Ventana del modelo (tokens):", m.max_seq_length)
ejemplo = "Los clubes adscritos al Campeonato Nacional de Liga de Primera División podrán inscribir en su primer equipo, hasta un máximo de tres futbolistas extranjeros no comunitarios pudiendo alinearse simultáneamente."
print("Tokens de una frase de 215 caracteres:", len(m.tokenizer(ejemplo)['input_ids']))
del m

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 721.42it/s]


Ventana del modelo (tokens): 128
Tokens de una frase de 215 caracteres: 49


## 1. Normas de Competición 2026/27 → un chunk por apartado, con su artículo

Estrategia: leer línea por línea, descartar el encabezado repetido y el número de página, y abrir un artículo nuevo
cada vez que aparece `ARTÍCULO N. – TÍTULO`. Cada chunk lleva delante `Art. N – TÍTULO` para que, aunque sea un
apartado suelto, el vector sepa de qué trata. `division` evita que las normas de Primera y Segunda (casi idénticas) se mezclen.

In [3]:
ENCAB = {"CAMPEONATOS NACIONALES DE", "LIGA DE PRIMERA Y SEGUNDA DIVISIÓN"}
ART = re.compile(r'^ARTÍCULO\s+(\d+)\.\s*[–-]\s*(.*)$')
TIT = re.compile(r'^T[ÍI]TULO\s+([IVX]+)$')
DISP = re.compile(r'^DISPOSICIONES (EXTRAORDINARIAS POR FUERZA MAYOR|FINALES)$')

def division(n: int) -> str:
    return 'primera' if 6 <= n <= 12 else 'segunda' if 13 <= n <= 19 else 'ambas'

def armar_cuerpo(lineas: list[str]) -> str:
    # Las líneas del PDF son cortes visuales: solo empieza párrafo nuevo en "1." / "a)" / "-".
    out = []
    for l in lineas:
        if not out or re.match(r'^(\d+\.|[a-f]\)|-)\s', l): out.append(l)
        else: out[-1] += ' ' + l
    return '\n\n'.join(out)

doc = pymupdf.open(RUTA_NORMAS)
print("Página 2 (debe ser el índice):", repr(doc[1].get_text()[:40]))
lineas = [(i + 1, l.strip()) for i in range(2, len(doc))            # saltamos portada e índice
          for l in doc[i].get_text().split('\n')
          if l.strip() and l.strip() not in ENCAB and not l.strip().isdigit()]

arts, actual, k = [], None, 0
while k < len(lineas):
    pag, s = lineas[k]
    if (mt := ART.match(s)):
        nombre = mt.group(2)
        if not nombre.endswith('.'):                                  # título partido en dos líneas
            k += 1; nombre += ' ' + lineas[k][1]
        actual = dict(art=int(mt.group(1)), nombre=nombre.rstrip('.'), pag=pag, texto=[])
        arts.append(actual)
    elif DISP.match(s):                                               # disposiciones finales: no son del Art. 34
        actual = dict(art=0, nombre=s.capitalize(), pag=pag, texto=[]); arts.append(actual)
    elif actual and not TIT.match(s):
        actual['texto'].append(s)
    k += 1

docs_normas = []
for a in arts:
    cab = f"Art. {a['art']} – {a['nombre']}" if a['art'] else a['nombre']
    for ch in splitter.split_text(armar_cuerpo(a['texto'])):
        docs_normas.append(dict(texto=f"{cab}\n{ch.lstrip('. ')}", documento='normas_competicion', seccion=cab,
                                pagina=a['pag'], division=division(a['art']) if a['art'] else 'ambas', articulo=a['art']))

print(len(arts), "secciones;", len(docs_normas), "chunks;", "máx", max(len(d['texto']) for d in docs_normas), "caracteres")
faltan = sorted(set(range(1, 35)) - {a['art'] for a in arts})
print("Artículos que faltan (debe ser []):", faltan)

Página 2 (debe ser el índice): 'índice\nTÍTULO I DISPOSICIONES GENERALES '
36 secciones; 124 chunks; máx 588 caracteres
Artículos que faltan (debe ser []): []


## 2. Reglas de Juego → capítulo por página con un mapa

Solo 110 de las 276 páginas traen el encabezado `Reglas de Juego 2026/27 | Regla N | Título`. Las demás heredan el capítulo
de la **portada de capítulo**, que está una página antes de la primera página con encabezado. `bisect_right` busca, para
cada página, el último capítulo que empezó en o antes de ella.

In [4]:
d = pymupdf.open(RUTA_REGLAS)
patron = re.compile(r'Reglas de Juego 2026/27\s*\|\s*([^\n]+)')
encabezados = {}                                                      # página (1-based) -> "Regla 11 | El fuera de juego"
for i, p in enumerate(d):
    mt = patron.search(p.get_text())
    if mt: encabezados[i + 1] = re.sub(r'\s+', ' ', mt.group(1)).strip()
print(len(encabezados), "de", len(d), "páginas traen encabezado")

primeras = {}
for pag, h in encabezados.items(): primeras.setdefault(h, pag)         # primera página de cada capítulo
inicios = sorted((pag - 1, h) for h, pag in primeras.items())          # portada = 1 página antes
paginas = [p for p, _ in inicios]

def seccion_de(n: int) -> str:
    i = bisect_right(paginas, n) - 1
    return inicios[i][1] if i >= 0 else 'Preliminares'

# PRUEBA: en las páginas que SÍ traen encabezado propio, el mapa debe coincidir con él.
malas = [(p, h, seccion_de(p)) for p, h in encabezados.items() if seccion_de(p) != h]
print(len(malas), "inconsistencias de", len(encabezados), "->", malas[:5])
for p in (48, 61, 62, 100, 164, 165):
    print(f"p.{p}: {seccion_de(p)}")

110 de 276 páginas traen encabezado
0 inconsistencias de 110 -> []
p.48: Regla 1 | El terreno de juego
p.61: Regla 2 | El balón
p.62: Regla 3 | Los jugadores
p.100: Regla 7 | La duración del partido
p.164: Protocolo del VAR
p.165: Protocolo del VAR


In [5]:
EXCLUIR = {'Índice', 'Preliminares'}                                    # tablas de contenido: solo meten ruido al buscar
docs_reglas, excluidas = [], 0
for n, pagina in enumerate(d, start=1):
    sec = seccion_de(n)
    texto = normalizar_espacios(limpiar_reglas(pagina.get_text()))
    if sec in EXCLUIR or len(texto) < 80:
        excluidas += 1; continue
    for ch in splitter.split_text(texto):
        if len(ch) >= 80:
            docs_reglas.append(dict(texto=f"[{sec}]\n{ch}", documento='reglas_juego', seccion=sec,
                                    pagina=n, division='', articulo=0))
print(len(docs_reglas), "chunks;", excluidas, "páginas excluidas")
print(docs_reglas[len(docs_reglas)//2]['texto'][:300])

678 chunks; 75 páginas excluidas
[Regla 12 | Faltas y conducta incorrecta]
134
 Todo jugador que arremeta contra un adversario en la disputa del balón de 
frente, por el costado o por detrás, utilizando una o ambas piernas con fuerza 
excesiva o poniendo en peligro la integridad física del adversario, estará 
jugando con excesiva d


## 3. Wikipedia → rescatamos los chunks ya limpios del índice viejo

Esos dos documentos ya están limpios y validados, así que no repetimos su limpieza: los leemos de la colección vieja
(solo lectura) y los partimos otra vez en trozos de ≤500 caracteres.

In [6]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='paraphrase-multilingual-MiniLM-L12-v2')
vieja = chromadb.PersistentClient(path=INDICE_VIEJO).get_collection('corpus_laliga', embedding_function=embedding_fn)
wiki = vieja.get(where={'documento': {'$in': ['primera_division', 'clasico']}}, include=['documents', 'metadatas'])

docs_wiki = []
for t, mt in zip(wiki['documents'], wiki['metadatas']):
    for ch in splitter.split_text(t):
        if len(ch) >= 80:
            docs_wiki.append(dict(texto=ch, documento=mt['documento'], seccion='', pagina=0, division='', articulo=0))
print(len(wiki['documents']), "chunks viejos ->", len(docs_wiki), "chunks nuevos")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 571.16it/s]


450 chunks viejos -> 871 chunks nuevos


## 4. Indexar en una colección nueva (el índice viejo no se toca)

In [7]:
todos = docs_normas + docs_reglas + docs_wiki
print("Total de chunks:", len(todos))

client = chromadb.PersistentClient(path=INDICE_NUEVO)
try: client.delete_collection(COLECCION_NUEVA)                         # permite re-ejecutar la celda sin error
except Exception: pass
col = client.create_collection(COLECCION_NUEVA, embedding_function=embedding_fn, metadata={"hnsw:space": "cosine"})

CAMPOS = ('documento', 'seccion', 'pagina', 'division', 'articulo')
for i in range(0, len(todos), 500):
    lote = todos[i:i + 500]
    col.add(ids=[f"c{i + j}" for j in range(len(lote))],
            documents=[c['texto'] for c in lote],
            metadatas=[{k: c[k] for k in CAMPOS} for c in lote])
print("Indexados:", col.count())

Total de chunks: 1673
Indexados: 1673


## 5. Medir la recuperación: ¿la fuente correcta aparece en el top-4?

Cada pregunta lleva el documento esperado y un fragmento del nombre de la sección. **Hit@1** = la fuente correcta es el
primer resultado; **Hit@4** = aparece entre los 4. Esa cifra va al capítulo de resultados de la tesis, y sirve para decidir
con datos (¿hace falta el Libro IV? ¿otro modelo de embeddings?) en lugar de intuición.

In [8]:
PREGUNTAS = [
    ("¿Cuántos suplentes se pueden usar en un partido?",              'normas_competicion', ['Art. 25']),
    ("¿Cuántas sustituciones puede hacer cada equipo?",               'normas_competicion', ['Art. 25']),
    ("¿Cómo funciona la sustitución por conmoción cerebral?",         'normas_competicion', ['Art. 25']),
    ("¿Cuántos equipos descienden de Primera División?",              'normas_competicion', ['Art. 8']),
    ("¿Cuántos puntos se dan por una victoria?",                      'normas_competicion', ['Art. 7']),
    ("¿Cuántos equipos participan en Primera División?",              'normas_competicion', ['Art. 6']),
    ("¿Cómo se desempata si dos equipos terminan con los mismos puntos?", 'normas_competicion', ['Art. 10', 'Art. 17']),
    ("¿Cuántas amonestaciones llevan a una suspensión?",              'normas_competicion', ['Art. 34']),
    ("¿Cuántos futbolistas extranjeros no comunitarios puede inscribir un club?", 'normas_competicion', ['Art. 22']),
    ("¿Cuándo abre el periodo de inscripción de licencias de invierno?", 'normas_competicion', ['Art. 23']),
    ("¿En qué horarios se juegan los partidos del sábado?",           'normas_competicion', ['Art. 31']),
    ("¿Qué es el fuera de juego?",                                    'reglas_juego', ['Regla 11']),
    ("¿Cuánto dura un partido?",                                      'reglas_juego', ['Regla 7']),
    ("¿Cómo se ejecuta un penal?",                                    'reglas_juego', ['Regla 14', 'Regla 12']),
    ("¿Qué es un saque de esquina?",                                  'reglas_juego', ['Regla 17']),
    ("¿Para qué sirve el VAR?",                                       'reglas_juego', ['VAR']),
    ("¿Qué es El Clásico entre el Barcelona y el Real Madrid?",       'clasico', ['']),
]

def acierta(meta, doc_esperado, subcadenas):
    return meta['documento'] == doc_esperado and any(s in meta['seccion'] for s in subcadenas)

hit1 = hit4 = 0
for q, doc_esp, subs in PREGUNTAS:
    metas = col.query(query_texts=[q], n_results=4)['metadatas'][0]
    pos = next((i for i, mt in enumerate(metas) if acierta(mt, doc_esp, subs)), None)
    hit1 += pos == 0; hit4 += pos is not None
    marca = '✔' if pos is not None else '✘'
    print(f"{marca} pos={pos} | {q}")
    if pos is None: print("     obtuve:", [(mt['documento'], mt['seccion'][:30]) for mt in metas])
print(f"\nHit@1: {hit1}/{len(PREGUNTAS)}   Hit@4: {hit4}/{len(PREGUNTAS)}")

✘ pos=None | ¿Cuántos suplentes se pueden usar en un partido?
     obtuve: [('reglas_juego', 'Glosario'), ('reglas_juego', 'Cambios en las Reglas 2026/27'), ('reglas_juego', 'Regla 7 | La duración del part'), ('reglas_juego', 'Glosario')]
✔ pos=3 | ¿Cuántas sustituciones puede hacer cada equipo?
✔ pos=0 | ¿Cómo funciona la sustitución por conmoción cerebral?
✘ pos=None | ¿Cuántos equipos descienden de Primera División?
     obtuve: [('normas_competicion', 'Art. 6 – EQUIPOS PARTICIPANTES'), ('primera_division', ''), ('primera_division', ''), ('normas_competicion', 'Art. 15 – CONSECUENCIAS CLASIF')]
✘ pos=None | ¿Cuántos puntos se dan por una victoria?
     obtuve: [('primera_division', ''), ('clasico', ''), ('primera_division', ''), ('primera_division', '')]
✔ pos=0 | ¿Cuántos equipos participan en Primera División?
✔ pos=2 | ¿Cómo se desempata si dos equipos terminan con los mismos puntos?
✘ pos=None | ¿Cuántas amonestaciones llevan a una suspensión?
     obtuve: [('reglas_juego', 'Glo

## 6. Búsqueda híbrida: embeddings + BM25 con cupos por fuente

**Por qué:** con solo embeddings (modelo de 128 tokens) el Hit@4 fue 13/17. El diagnóstico mostró que el modelo pone el
Art. 25 ("suplentes") en la posición 21 o peor, mientras que BM25 lo pone primero. Las palabras exactas ("suplentes",
"amonestaciones") las captura mejor la búsqueda léxica; el significado, el embedding.

**Cómo:** `buscar_hibrido` junta los dos rankings con *Reciprocal Rank Fusion* (cada ranking aporta 1/(60 + posición); se
suman posiciones, no puntajes, porque están en escalas distintas). `buscar_con_cupos` busca por separado en cada fuente
(3 de Normas, 3 de Reglas de Juego, 2 de Wikipedia) para que una fuente grande o ruidosa no ocupe todo el contexto.

In [19]:
import unicodedata
import snowballstemmer
from rank_bm25 import BM25Okapi

stemmer = snowballstemmer.stemmer('spanish')
# Palabras vacías: no distinguen un artículo de otro y solo diluyen el puntaje BM25. (Sin acentos: ya se los quitamos.)
STOP = set("de la el en y a los las del un una por con para que se es al lo como su cual cuales cuantos cuantas "
           "cuanto cuando donde qui quien".split())

def tokenizar(t: str) -> list[str]:
    # Sin acentos y en minúsculas, para que "sustitución" y "sustituciones" compartan raíz tras el stemmer.
    t = unicodedata.normalize('NFD', t.lower())
    t = ''.join(c for c in t if unicodedata.category(c) != 'Mn')
    return [stemmer.stemWord(w) for w in re.findall(r'[a-z0-9]+', t) if w not in STOP]

# Un índice BM25 por fuente (y uno global para documentos=None). El IDF mide "qué tan rara es una palabra" DENTRO del
# conjunto indexado: si "suplentes" aparece mil veces en Reglas de Juego, en un índice global parece común y pesa poco,
# aunque en Normas sea justo la palabra que identifica el Art. 25.
_bm25: dict = {}
def indice_bm25(documentos):
    clave = tuple(documentos) if documentos else None
    if clave not in _bm25:
        idx = [i for i, c in enumerate(todos) if clave is None or c['documento'] in clave]   # posiciones en `todos`
        _bm25[clave] = (idx, BM25Okapi([tokenizar(todos[i]['texto']) for i in idx]))
    return _bm25[clave]

def buscar_hibrido(q: str, k: int = 4, candidatos: int = 20, k_rrf: int = 60,
                   documentos: tuple[str, ...] | None = None, denso: bool = True, lexico: bool = True) -> list[dict]:
    """Fusiona el ranking por significado (embeddings) con el ranking por palabras (BM25) usando RRF."""
    rankings = []
    if denso:
        r = col.query(query_texts=[q], n_results=candidatos,
                      where={'documento': {'$in': list(documentos)}} if documentos else None)
        rankings.append([int(i[1:]) for i in r['ids'][0]])          # "c123" -> 123
    if lexico:
        idx, bm = indice_bm25(documentos)
        sc = bm.get_scores(tokenizar(q))
        orden = [idx[j] for j in sorted((j for j in range(len(idx)) if sc[j] > 0), key=lambda j: -sc[j])]
        rankings.append(orden[:candidatos])
    # RRF: cada ranking aporta 1/(k_rrf + posición). No mezcla puntajes (que están en escalas distintas), solo posiciones.
    fusion: dict[int, float] = {}
    for orden in rankings:
        for pos, i in enumerate(orden, start=1):
            fusion[i] = fusion.get(i, 0.0) + 1.0 / (k_rrf + pos)
    return [todos[i] for i in sorted(fusion, key=fusion.get, reverse=True)[:k]]

GRUPOS = [('normas_competicion',), ('reglas_juego',), ('primera_division', 'clasico')]

def buscar_por_fuente(q: str, k: int = 4) -> list[dict]:
    # Buscamos dentro de cada fuente por separado: así una fuente enorme (Reglas de Juego) no tapa a una pequeña (Normas).
    listas = [buscar_hibrido(q, k=k, documentos=g) for g in GRUPOS]
    salida = []
    for pos in range(k):                      # por turnos: 1.º de cada fuente, luego 2.º de cada fuente...
        for lista in listas:
            if pos < len(lista): salida.append(lista[pos])
    return salida[:k]

In [20]:
def evaluar(buscar, nombre: str) -> None:
    hit1 = hitk = 0
    for q, doc_esp, subs in PREGUNTAS:
        metas = buscar(q)
        pos = next((i for i, mt in enumerate(metas) if acierta(mt, doc_esp, subs)), None)
        hit1 += pos == 0; hitk += pos is not None
        if pos is None: print(f"   ✘ {q}\n     obtuve:", [(mt['documento'], mt['seccion'][:25]) for mt in metas])
    print(f"{nombre:<26} Hit@1: {hit1}/{len(PREGUNTAS)}   Hit@{len(metas)}: {hitk}/{len(PREGUNTAS)}\n")

evaluar(lambda q: col.query(query_texts=[q], n_results=4)['metadatas'][0], 'solo embeddings')
evaluar(lambda q: buscar_hibrido(q, denso=False), 'solo BM25')
evaluar(buscar_hibrido, 'híbrido (RRF)')

   ✘ ¿Cuántos suplentes se pueden usar en un partido?
     obtuve: [('reglas_juego', 'Glosario'), ('reglas_juego', 'Cambios en las Reglas 202'), ('reglas_juego', 'Regla 7 | La duración del'), ('reglas_juego', 'Glosario')]
   ✘ ¿Cuántos equipos descienden de Primera División?
     obtuve: [('normas_competicion', 'Art. 6 – EQUIPOS PARTICIP'), ('primera_division', ''), ('primera_division', ''), ('normas_competicion', 'Art. 15 – CONSECUENCIAS C')]
   ✘ ¿Cuántos puntos se dan por una victoria?
     obtuve: [('primera_division', ''), ('clasico', ''), ('primera_division', ''), ('primera_division', '')]
   ✘ ¿Cuántas amonestaciones llevan a una suspensión?
     obtuve: [('reglas_juego', 'Glosario'), ('reglas_juego', 'Regla 7 | La duración del'), ('reglas_juego', 'Cambios en las Reglas 202'), ('reglas_juego', 'Regla 5 | El árbitro')]
solo embeddings            Hit@1: 8/17   Hit@4: 13/17

   ✘ ¿Cuántos suplentes se pueden usar en un partido?
     obtuve: [('reglas_juego', 'Regla 5 | El árbitro')

In [21]:
evaluar(buscar_por_fuente, 'híbrido por fuente k=4')
evaluar(lambda q: buscar_por_fuente(q, k=6), 'híbrido por fuente k=6')

   ✘ ¿Cuántos suplentes se pueden usar en un partido?
     obtuve: [('normas_competicion', 'Art. 28 – UNIFORMIDAD'), ('reglas_juego', 'Regla 3 | Los jugadores'), ('primera_division', ''), ('normas_competicion', 'Art. 10 – DETERMINACIÓN D')]
   ✘ ¿Cuántos equipos descienden de Primera División?
     obtuve: [('normas_competicion', 'Art. 6 – EQUIPOS PARTICIP'), ('reglas_juego', 'Regla 3 | Los jugadores'), ('primera_division', ''), ('normas_competicion', 'Art. 22 – NÚMERO DE LICEN')]
   ✘ ¿Qué es un saque de esquina?
     obtuve: [('normas_competicion', 'Art. 28 – UNIFORMIDAD'), ('reglas_juego', 'Cambios en las Reglas 202'), ('clasico', ''), ('normas_competicion', 'Art. 28 – UNIFORMIDAD')]
híbrido por fuente k=4     Hit@1: 9/17   Hit@4: 14/17

   ✘ ¿Cuántos suplentes se pueden usar en un partido?
     obtuve: [('normas_competicion', 'Art. 28 – UNIFORMIDAD'), ('reglas_juego', 'Regla 3 | Los jugadores'), ('primera_division', ''), ('normas_competicion', 'Art. 10 – DETERMINACIÓN D'), ('reglas

In [22]:
def diagnostico(q: str, art: int, fuente=('normas_competicion',)) -> None:
    """¿En qué posición queda el artículo correcto dentro de cada ranking? (candidatos=60 para verlo aunque quede lejos)"""
    def posiciones(**kw):
        lista = buscar_hibrido(q, k=60, candidatos=60, documentos=fuente, **kw)
        return [p for p, c in enumerate(lista, 1) if c['seccion'].startswith(f"Art. {art} ")]
    print(f"{q}\n   Art. {art} -> solo embeddings: {posiciones(lexico=False)} | solo BM25: {posiciones(denso=False)} | híbrido: {posiciones()}")

for q, art in [("¿Cuántos suplentes se pueden usar en un partido?", 25),
               ("¿Cuántos equipos descienden de Primera División?", 8),
               ("¿Cuántos puntos se dan por una victoria?", 7),
               ("¿Cuántas amonestaciones llevan a una suspensión?", 34)]:
    diagnostico(q, art)

# ¿Qué dice realmente el Art. 25? (para ver si usa la palabra "suplentes")
for c in todos:
    if c['seccion'].startswith("Art. 25 "):
        print("\n", c['texto'][:350].replace("\n", " "))

¿Cuántos suplentes se pueden usar en un partido?
   Art. 25 -> solo embeddings: [21, 24, 33, 34, 49, 54, 59] | solo BM25: [1, 5, 6, 7, 9, 11, 12, 13, 15, 16, 17, 18] | híbrido: [3, 5, 10, 13, 14, 18, 19, 37, 40, 45, 49, 51]
¿Cuántos equipos descienden de Primera División?
   Art. 8 -> solo embeddings: [10, 25, 51] | solo BM25: [8, 11, 19] | híbrido: [6, 21, 25]
¿Cuántos puntos se dan por una victoria?
   Art. 7 -> solo embeddings: [2] | solo BM25: [2] | híbrido: [1]
¿Cuántas amonestaciones llevan a una suspensión?
   Art. 34 -> solo embeddings: [1, 2, 3] | solo BM25: [1, 2, 3] | híbrido: [1, 2, 3]

 Art. 25 – SUPLENTES Y SUSTITUCIONES 1. El número de futbolistas eventualmente suplentes no podrá exceder, en ningún caso, de doce futbolistas.

 Art. 25 – SUPLENTES Y SUSTITUCIONES 2. La intervención de estos en un partido será, como máximo, de cinco futbolistas suplentes mediante la sustitución de un futbolista, como límite, en tres interrupciones del partido por cada equipo previa autoriz

In [23]:
CUPOS = {('normas_competicion',): 3, ('reglas_juego',): 3, ('primera_division', 'clasico'): 2}

def buscar_con_cupos(q: str) -> list[dict]:
    # Cada fuente aporta sus mejores N: una fuente grande o ruidosa ya no puede ocupar todo el contexto.
    salida: list[dict] = []
    for fuente, cupo in CUPOS.items():
        salida += buscar_hibrido(q, k=cupo, documentos=fuente)
    return salida

evaluar(lambda q: col.query(query_texts=[q], n_results=8)['metadatas'][0], 'solo embeddings (k=8)')
evaluar(lambda q: buscar_hibrido(q, k=8), 'híbrido global (k=8)')
evaluar(buscar_con_cupos, 'híbrido con cupos (k=8)')

   ✘ ¿Cuántos suplentes se pueden usar en un partido?
     obtuve: [('reglas_juego', 'Glosario'), ('reglas_juego', 'Cambios en las Reglas 202'), ('reglas_juego', 'Regla 7 | La duración del'), ('reglas_juego', 'Glosario'), ('reglas_juego', 'Regla 3 | Los jugadores'), ('reglas_juego', 'Regla 4 | El equipamiento'), ('normas_competicion', 'Art. 10 – DETERMINACIÓN D'), ('normas_competicion', 'Art. 17 – DETERMINACIÓN D')]
   ✘ ¿Cuántos equipos descienden de Primera División?
     obtuve: [('normas_competicion', 'Art. 6 – EQUIPOS PARTICIP'), ('primera_division', ''), ('primera_division', ''), ('normas_competicion', 'Art. 15 – CONSECUENCIAS C'), ('reglas_juego', 'Regla 3 | Los jugadores'), ('normas_competicion', 'Art. 24 – ALINEACIÓN DE F'), ('normas_competicion', 'Art. 14 – SISTEMA DE COMP'), ('normas_competicion', 'Art. 13 – EQUIPOS PARTICI')]
solo embeddings (k=8)      Hit@1: 8/17   Hit@8: 15/17

   ✘ ¿Cuántos suplentes se pueden usar en un partido?
     obtuve: [('reglas_juego', 'Regla 3 |

### Resultados y limitaciones

| Método | Hit@1 | Hit@k |
|---|---|---|
| solo embeddings | 8/17 | 13/17 (k=4) · 15/17 (k=8) |
| solo BM25 | 9/17 | 11/17 (k=4) |
| híbrido (RRF) | 10/17 | 14/17 (k=4) · 15/17 (k=8) |
| híbrido con cupos | 9/17* | 16/17 (k=8) |

\*Con cupos el orden es por fuente, no por relevancia, así que su Hit@1 no es comparable.

- Con 17 preguntas, una acertada de diferencia no es concluyente, y el método se ajustó mirando estas mismas preguntas.
  La cifra para la tesis debe medirse con preguntas nuevas, evaluadas una sola vez.
- Fallo conocido: "¿Cuántos equipos descienden…?" (Art. 8). El texto dice "descenso/descenderán" y el stemmer no une
  "descienden" con ellas (verbo irregular). Mejora futura: reescribir la consulta antes de buscar.
- Mejora futura: probar un modelo de embeddings multilingüe más grande (el actual ubica el Art. 25 en la posición 21+).